# FinGuard Fraud Detection Pipeline
## Notebook 02 — Silver Layer Transformations

Cleans, deduplicates, enriches, and standardises Bronze data. No business
aggregations yet — that happens in Gold.

Source: `finguard.bronze.*`
Target: `finguard.silver.*`

## Imports

In [0]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    BooleanType,
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    TimestampType,
)
from delta.tables import DeltaTable

print(f"Spark version   : {spark.version}")
print(f"Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} AEST")

## Configuration

In [0]:
CATALOG_NAME   = "finguard"
BRONZE_SCHEMA  = "bronze"
SILVER_SCHEMA  = "silver"
MONITOR_SCHEMA = "monitoring"

BRONZE_TRANSACTIONS = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.transactions"
BRONZE_CUSTOMERS    = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.customers"
BRONZE_MERCHANTS    = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.merchants"

SILVER_TRANSACTIONS_CLEANED  = f"{CATALOG_NAME}.{SILVER_SCHEMA}.transactions_cleaned"
SILVER_TRANSACTIONS_ENRICHED = f"{CATALOG_NAME}.{SILVER_SCHEMA}.transactions_enriched"
SILVER_CUSTOMERS             = f"{CATALOG_NAME}.{SILVER_SCHEMA}.customers"

DEAD_LETTER_TABLE = f"{CATALOG_NAME}.{MONITOR_SCHEMA}.dead_letter"

BATCH_ID      = f"silver_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
PIPELINE_NAME = "finguard_silver_transformation"

AUSTRAC_THRESHOLD = 10_000.00

print(f"Batch ID : {BATCH_ID}")
print(f"Source   : {CATALOG_NAME}.{BRONZE_SCHEMA}.*")
print(f"Target   : {CATALOG_NAME}.{SILVER_SCHEMA}.*")

## Create Silver and Monitoring Schemas

In [0]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SILVER_SCHEMA}
    COMMENT 'FinGuard Silver layer — cleaned and enriched Delta tables'
""")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{MONITOR_SCHEMA}
    COMMENT 'FinGuard pipeline monitoring — DQ results and dead-letter rows'
""")

print(f"✓ Schema ready: {CATALOG_NAME}.{SILVER_SCHEMA}")
print(f"✓ Schema ready: {CATALOG_NAME}.{MONITOR_SCHEMA}")

## Read Bronze Tables

In [0]:
txn_bronze       = spark.table(BRONZE_TRANSACTIONS)
customers_bronze = spark.table(BRONZE_CUSTOMERS)
merchants_bronze = spark.table(BRONZE_MERCHANTS)

# Strip Bronze audit columns before transforming — Silver adds its own
audit_cols = ["_ingested_at", "_source_file", "_batch_id", "_pipeline_name"]

txn_raw       = txn_bronze.drop(*audit_cols)
customers_raw = customers_bronze.drop(*audit_cols)
merchants_raw = merchants_bronze.drop(*audit_cols)

print(f"Bronze transactions : {txn_raw.count():>10,} rows")
print(f"Bronze customers    : {customers_raw.count():>10,} rows")
print(f"Bronze merchants    : {merchants_raw.count():>10,} rows")

## Type Casting & Standardisation

Bronze lands `txn_timestamp` and `txn_date` as strings; cast here, along
with derived columns used in later fraud feature engineering.

In [0]:
txn_typed = (
    txn_raw
    .withColumn(
        "amount",
        F.regexp_replace(F.col("amount").cast("string"), "[^0-9.]", "").cast(DoubleType())
    )
    .withColumn("txn_timestamp", F.to_timestamp(F.col("txn_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("txn_date",      F.to_date(F.col("txn_date"), "yyyy-MM-dd"))
    .withColumn("currency",      F.upper(F.col("currency")))
    .withColumn("channel",       F.lower(F.col("channel")))
    .withColumn("card_network",  F.initcap(F.col("card_network")))
    .withColumn("device_type",   F.lower(F.col("device_type")))
    .withColumn("ip_country",    F.upper(F.col("ip_country")))
    .withColumn("txn_year",      F.year(F.col("txn_timestamp")))
    .withColumn("txn_quarter",   F.quarter(F.col("txn_timestamp")))
    .withColumn("txn_week",      F.weekofyear(F.col("txn_timestamp")))
    .withColumn("is_weekend",    F.dayofweek(F.col("txn_timestamp")).isin([1, 7]))
    .withColumn("is_night",      F.col("txn_hour").between(22, 23) |
                                  F.col("txn_hour").between(0, 5))
    .withColumn(
        "amount_band",
        F.when(F.col("amount") < 50,    F.lit("micro"))
         .when(F.col("amount") < 200,   F.lit("small"))
         .when(F.col("amount") < 1_000, F.lit("medium"))
         .when(F.col("amount") < 5_000, F.lit("large"))
         .otherwise(F.lit("very_large"))
    )
    .withColumn(
        "is_near_austrac_threshold",
        F.col("amount").between(AUSTRAC_THRESHOLD * 0.90, AUSTRAC_THRESHOLD)
    )
)

customers_typed = (
    customers_raw
    .withColumn("date_of_birth",     F.to_date(F.col("date_of_birth"), "yyyy-MM-dd"))
    .withColumn("account_open_date", F.to_date(F.col("account_open_date"), "yyyy-MM-dd"))
    .withColumn("address_state",     F.upper(F.col("address_state")))
    .withColumn("employment_status", F.lower(F.col("employment_status")))
    .withColumn(
        "age_years",
        F.floor(F.datediff(F.current_date(), F.col("date_of_birth")) / 365).cast(IntegerType())
    )
    .withColumn("account_tenure_days", F.datediff(F.current_date(), F.col("account_open_date")))
    .withColumn(
        "income_band",
        F.when(F.col("annual_income_aud") < 50_000,  F.lit("low"))
         .when(F.col("annual_income_aud") < 100_000, F.lit("medium"))
         .when(F.col("annual_income_aud") < 200_000, F.lit("high"))
         .otherwise(F.lit("very_high"))
    )
)

print("✓ Type casting complete")
print(f"  Transaction columns : {len(txn_typed.columns)}")
print(f"  Customer columns    : {len(customers_typed.columns)}")

## Deduplication

Payment gateways can resend the same transaction on retry. We keep the
latest version of each `transaction_id`, using a Window rather than
`dropDuplicates` so we control which duplicate survives.

In [0]:
dedup_window = Window.partitionBy("transaction_id").orderBy(F.desc("txn_timestamp"))

txn_deduped = (
    txn_typed
    .withColumn("_row_num", F.row_number().over(dedup_window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

total_before = txn_typed.count()
total_after  = txn_deduped.count()

print(f"Before deduplication : {total_before:,}")
print(f"After deduplication  : {total_after:,}")
print(f"Duplicates removed   : {total_before - total_after:,}")

## Null Handling Strategy

Critical columns route to a dead-letter table rather than being silently
dropped. Non-critical nulls get a deliberate default per column type.

In [0]:
CRITICAL_COLUMNS = ["transaction_id", "customer_id", "merchant_id", "amount"]

bad_row_filter = (
    F.col("transaction_id").isNull() |
    F.col("customer_id").isNull() |
    F.col("merchant_id").isNull() |
    F.col("amount").isNull() |
    (F.col("amount") <= 0)
)

bad_rows = (
    txn_deduped.filter(bad_row_filter)
    .withColumn(
        "_rejection_reason",
        F.when(F.col("transaction_id").isNull(), F.lit("null_transaction_id"))
         .when(F.col("customer_id").isNull(), F.lit("null_customer_id"))
         .when(F.col("merchant_id").isNull(), F.lit("null_merchant_id"))
         .when(F.col("amount").isNull(), F.lit("null_amount"))
         .when(F.col("amount") <= 0, F.lit("invalid_amount"))
         .otherwise(F.lit("unknown"))
    )
    .withColumn("_rejected_at",    F.current_timestamp())
    .withColumn("_batch_id",       F.lit(BATCH_ID))
    .withColumn("_pipeline_name",  F.lit(PIPELINE_NAME))
)

bad_row_count = bad_rows.count()

if bad_row_count > 0:
    bad_rows.write.format("delta").mode("append").saveAsTable(DEAD_LETTER_TABLE)
    print(f"⚠️  {bad_row_count:,} bad rows quarantined → {DEAD_LETTER_TABLE}")
else:
    print("✓ No bad rows found — dead-letter table empty")

txn_clean = txn_deduped.filter(~bad_row_filter)
print(f"✓ Clean transactions: {txn_clean.count():,}")

In [0]:
txn_clean = (
    txn_clean
    .fillna({
        "channel":          "unknown",
        "card_network":     "unknown",
        "device_type":      "unknown",
        "ip_country":       "unknown",
        "merchant_category": "unknown",
        "mcc_code":         "9999",
        "response_code":    "00",
        "txn_day_of_week":  "unknown",
        "fraud_type":       "none",
        "fraud_indicator":  "none",
    })
    .fillna({
        "is_declined":     False,
        "is_international": False,
        "is_fraud":        False,
        "is_weekend":      False,
        "is_night":        False,
        "is_near_austrac_threshold": False,
    })
    .fillna({"txn_hour": 0})
)

print("✓ Null filling complete")

## Broadcast Joins

`customers` (~ 5,000 rows) and `merchants` (~800 rows) are small enough to
broadcast to every executor, avoiding a shuffle against the much larger
transactions table.

In [0]:
customers_dim = customers_typed.select(
    F.col("customer_id"),
    F.col("address_state").alias("customer_state"),
    F.col("address_suburb").alias("customer_suburb"),
    F.col("annual_income_aud"),
    F.col("employment_status"),
    F.col("credit_score"),
    F.col("age_years"),
    F.col("income_band"),
    F.col("account_tenure_days"),
    F.col("is_high_risk"),
    F.col("kyc_verified"),
)

merchants_dim = merchants_raw.select(
    F.col("merchant_id"),
    F.col("merchant_name"),
    F.col("category").alias("merchant_category_detail"),
    F.col("country").alias("merchant_country"),
    F.col("is_international").alias("merchant_is_international"),
    F.col("is_online_only").alias("merchant_is_online"),
    F.col("risk_level").alias("merchant_risk_level"),
    F.col("abn"),
)

print(f"customers_dim columns : {len(customers_dim.columns)}")
print(f"merchants_dim columns : {len(merchants_dim.columns)}")

In [0]:
txn_enriched = (
    txn_clean
    .join(F.broadcast(customers_dim), on="customer_id", how="left")
    .join(F.broadcast(merchants_dim), on="merchant_id", how="left")
)

# Left joins can introduce nulls when a customer/merchant has no dimension match
txn_enriched = txn_enriched.fillna({
    "customer_state":           "unknown",
    "customer_suburb":          "unknown",
    "income_band":              "unknown",
    "employment_status":        "unknown",
    "merchant_name":            "unknown",
    "merchant_risk_level":      "unknown",
    "merchant_country":         "unknown",
    "merchant_is_international": False,
    "merchant_is_online":        False,
    "is_high_risk":              False,
    "kyc_verified":              True,
})

print(f"✓ Enriched transactions: {txn_enriched.count():,} rows")
print(f"  Total columns after join: {len(txn_enriched.columns)}")

## Window Functions

Rolling spend, velocity, and ranking features — all calculated per customer
using PySpark window functions, partitioned by `customer_id`.

In [0]:
window_7d = (
    Window.partitionBy("customer_id")
    .orderBy(F.col("txn_timestamp").cast("long"))
    .rangeBetween(-604_800, 0)  # 7 days, in seconds
)

window_30d = (
    Window.partitionBy("customer_id")
    .orderBy(F.col("txn_timestamp").cast("long"))
    .rangeBetween(-2_592_000, 0)  # 30 days, in seconds
)

window_rank = Window.partitionBy("customer_id").orderBy(F.col("txn_timestamp").cast("long"))

print("✓ Window specs defined")

In [0]:
txn_windowed = (
    txn_enriched
    .withColumn("spend_7d_aud",  F.sum("amount").over(window_7d))
    .withColumn("spend_30d_aud", F.sum("amount").over(window_30d))
    .withColumn("txn_count_7d",  F.count("transaction_id").over(window_7d))
    .withColumn("txn_count_30d", F.count("transaction_id").over(window_30d))
    .withColumn("avg_amount_7d", F.round(F.avg("amount").over(window_7d), 2))

    .withColumn("prev_txn_timestamp", F.lag("txn_timestamp", 1).over(window_rank))
    .withColumn(
        "seconds_since_prev_txn",
        F.when(
            F.col("prev_txn_timestamp").isNotNull(),
            F.col("txn_timestamp").cast("long") - F.col("prev_txn_timestamp").cast("long")
        ).otherwise(F.lit(None))
    )
    .withColumn("is_rapid_succession", F.col("seconds_since_prev_txn") < 600)

    .withColumn("customer_txn_sequence", F.row_number().over(window_rank))
    .withColumn("is_first_transaction", F.col("customer_txn_sequence") == 1)
    .withColumn(
        "amount_vs_30d_avg",
        F.round(
            F.col("amount") / F.when(
                F.avg("amount").over(window_30d) != 0,
                F.avg("amount").over(window_30d)
            ).otherwise(F.lit(1.0)),
            2
        )
    )
    .drop("prev_txn_timestamp")
)

print(f"✓ Window functions applied")
print(f"  Total columns: {len(txn_windowed.columns)}")

## Add Silver Audit Columns & Write

In [0]:
def add_silver_audit_columns(df, batch_id, pipeline_name):
    """Add Silver-layer audit columns — when transformation was applied."""
    return (
        df
        .withColumn("_silver_processed_at", F.current_timestamp())
        .withColumn("_silver_batch_id",     F.lit(batch_id))
        .withColumn("_pipeline_name",       F.lit(pipeline_name))
    )

txn_final = add_silver_audit_columns(txn_windowed, BATCH_ID, PIPELINE_NAME)
print("✓ Audit columns added")

In [0]:
# transactions_cleaned: deduped, typed, null-handled — no joins yet
txn_clean_final = add_silver_audit_columns(txn_clean, BATCH_ID, PIPELINE_NAME)

(
    txn_clean_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("partition_date")
    .saveAsTable(SILVER_TRANSACTIONS_CLEANED)
)

count = spark.table(SILVER_TRANSACTIONS_CLEANED).count()
print(f"✓ {SILVER_TRANSACTIONS_CLEANED}: {count:,} rows")

In [0]:
# transactions_enriched: full join + window features — Gold layer reads from here
(
    txn_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("partition_date")
    .saveAsTable(SILVER_TRANSACTIONS_ENRICHED)
)

count = spark.table(SILVER_TRANSACTIONS_ENRICHED).count()
print(f"✓ {SILVER_TRANSACTIONS_ENRICHED}: {count:,} rows")

In [0]:
customers_final = add_silver_audit_columns(customers_typed, BATCH_ID, PIPELINE_NAME)

(
    customers_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("address_state")
    .saveAsTable(SILVER_CUSTOMERS)
)

count = spark.table(SILVER_CUSTOMERS).count()
print(f"✓ {SILVER_CUSTOMERS}: {count:,} rows")

## Data Quality Checks

In [0]:
silver_df = spark.table(SILVER_TRANSACTIONS_ENRICHED)
total     = silver_df.count()

print("Silver Layer — Data Quality Report")
print("═" * 50)
print(f"  Total rows             : {total:,}")

for col_name in CRITICAL_COLUMNS:
    null_count = silver_df.filter(F.col(col_name).isNull()).count()
    status = "✓" if null_count == 0 else "⚠️"
    print(f"  {status} Nulls in {col_name:<25}: {null_count:,}")

neg_amounts = silver_df.filter(F.col("amount") <= 0).count()
print(f"  {'✓' if neg_amounts == 0 else '⚠️'} Negative amounts           : {neg_amounts:,}")

null_window = silver_df.filter(F.col("spend_7d_aud").isNull()).count()
print(f"  {'✓' if null_window == 0 else '⚠️'} Null spend_7d_aud          : {null_window:,}")

fraud_rate = (silver_df.filter(F.col("is_fraud") == True).count() / total * 100) if total > 0 else 0
print(f"\n  Fraud rate             : {fraud_rate:.2f}%")

rapid = silver_df.filter(F.col("is_rapid_succession") == True).count()
print(f"  Rapid succession txns  : {rapid:,}  ({rapid/total*100:.2f}%)")

near_threshold = silver_df.filter(F.col("is_near_austrac_threshold") == True).count()
print(f"  Near AUSTRAC threshold : {near_threshold:,}")
print("═" * 50)

## OPTIMIZE Silver Tables

ZORDERing on `is_fraud` co-locates fraud rows in fewer files, since Gold's
fraud queries will always filter `WHERE is_fraud = TRUE`.

In [0]:
spark.sql(f"""
    OPTIMIZE {SILVER_TRANSACTIONS_ENRICHED}
    ZORDER BY (customer_id, txn_date, is_fraud)
""")

print(f"✓ OPTIMIZE complete: {SILVER_TRANSACTIONS_ENRICHED}")

## Summary

In [0]:
print("═" * 60)
print("  SILVER LAYER COMPLETE")
print("═" * 60)
print(f"  Batch ID : {BATCH_ID}")

silver_tables = [SILVER_TRANSACTIONS_CLEANED, SILVER_TRANSACTIONS_ENRICHED, SILVER_CUSTOMERS]
for table in silver_tables:
    count = spark.table(table).count()
    print(f"  {table:<50} {count:>10,} rows")

print("  Transformations applied:")
print("    ✓ Type casting (StringType → TimestampType, DateType)")
print("    ✓ Deduplication via Window row_number()")
print("    ✓ Dead-letter table for bad rows")
print("    ✓ Null handling strategy per column type")
print("    ✓ Broadcast joins (customers + merchants)")
print("    ✓ Window functions (7d/30d rolling, velocity, ranking)")
print("    ✓ Derived columns (amount_band, is_night, is_weekend)")
print("    ✓ OPTIMIZE + ZORDER BY (customer_id, txn_date, is_fraud)")

print("  Next → notebooks/03_gold_feature_engineering.ipynb")
print("═" * 60)